# LSTM v4 — Live Inference Notebook

This notebook demonstrates the full inference pipeline: fetch live data from APIs, 
build the input tensor, run the trained model, and show the 24-hour forecast with 
optimal charging window.

## APIs Used

| API | What it provides | Auth | Cost |
|-----|-----------------|------|------|
| **Open-Meteo** | Weather forecast (wind, temp) for next 7 days | None needed | Free |
| **Energi Data Service** | Past carbon intensity + day-ahead prices for Denmark | None needed | Free |
| **Electricity Maps** | Latest carbon intensity (optional fallback) | Free API key | Free tier |

## Setup

1. `pip install requests pandas numpy tensorflow`
2. Get a free Electricity Maps API key at https://www.electricitymaps.com/free-tier-api (optional)
3. Set environment variable: `ELECTRICITY_MAPS_API_KEY=your_key_here`


# Imports & Configuration

In [1]:
import os
import json
import requests
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta, timezone

print(f"TensorFlow: {tf.__version__}")


TensorFlow: 2.20.0


In [2]:
# ─── Paths to your trained model and scaler ───
MODEL_DIR  = "saved_model_lstm_v4"
SCALER_FILE = "scaler_params_v4.json"

# ─── API Keys (from environment variables) ───
# Only Electricity Maps needs a key — the others are free
EMAPS_API_KEY = os.environ.get("ELECTRICITY_MAPS_API_KEY", "")

# ─── Model config (must match training) ───
WINDOW_SIZE = 168         # 7 days of hourly history
FORECAST_HORIZON = 24     # predict next 24 hours

# ─── Location: Denmark (center point for weather) ───
DENMARK_LAT = 55.88
DENMARK_LON = 9.77

# ─── Feature list (must match training exactly) ───
FEATURE_NAMES = [
    "carbon_intensity",
    "diff_1",
    "diff_24",
    "hour_sin", "hour_cos",
    "year_sin", "year_cos",
    "dow_sin", "dow_cos",
    "wind_speed",
    "temperature",
    "price_eur_mwh",
]

print(f"Model dir: {MODEL_DIR}")
print(f"Features: {len(FEATURE_NAMES)}")
print(f"Electricity Maps key: {'set' if EMAPS_API_KEY else 'NOT SET (will use Energi Data Service only)'}")


Model dir: saved_model_lstm_v4
Features: 12
Electricity Maps key: NOT SET (will use Energi Data Service only)


# Load Trained Model & Scaler

In [3]:
# Load the trained model
model = tf.keras.models.load_model(os.path.join(MODEL_DIR, "final_model.keras"))
print(f"Model loaded: {model.input_shape} → {model.output_shape}")
print(f"Parameters: {model.count_params():,}")

# Load scaler parameters
with open(SCALER_FILE) as f:
    scaler = json.load(f)

feature_means = np.array(scaler["feature_means"])
feature_scales = np.array(scaler["feature_scales"])
target_mean = scaler["target_mean"][0]
target_scale = scaler["target_scale"][0]

print(f"Scaler loaded — target mean: {target_mean:.2f}, scale: {target_scale:.2f}")
print(f"Scaler features: {scaler['feature_names']}")


Model loaded: (None, 168, 12) → (None, 24)
Parameters: 58,456
Scaler loaded — target mean: 119.08, scale: 63.90
Scaler features: ['Carbon intensity gCO₂eq/kWh (direct)', 'diff_1', 'diff_24', 'hour_sin', 'hour_cos', 'year_sin', 'year_cos', 'dow_sin', 'dow_cos', 'wind_speed', 'temperature', 'price_eur_mwh']


# Fetch Weather Forecast (Open-Meteo)

Open-Meteo provides free weather forecasts — no API key needed. 
We use the DMI HARMONIE model for Denmark specifically.
Returns hourly wind speed and temperature for the past 7 days + next 2 days.


In [4]:
def fetch_weather(lat=DENMARK_LAT, lon=DENMARK_LON):
    """Fetch past 7 days + next 2 days of hourly weather from Open-Meteo."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,wind_speed_10m",
        "past_days": 7,
        "forecast_days": 2,
        "timezone": "UTC",
        "models": "dmi_seamless",    # Use DMI model for Denmark
    }
    
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    
    weather = pd.DataFrame({
        "Datetime (UTC)": pd.to_datetime(data["hourly"]["time"]),
        "wind_speed": data["hourly"]["wind_speed_10m"],
        "temperature": data["hourly"]["temperature_2m"],
    })
    
    print(f"Weather: {len(weather)} hours")
    print(f"  Range: {weather['Datetime (UTC)'].min()} to {weather['Datetime (UTC)'].max()}")
    print(f"  Wind speed: {weather['wind_speed'].min():.1f} - {weather['wind_speed'].max():.1f} m/s")
    print(f"  Temperature: {weather['temperature'].min():.1f} - {weather['temperature'].max():.1f} °C")
    
    return weather

weather_df = fetch_weather()
weather_df.tail()


Weather: 216 hours
  Range: 2026-03-03 00:00:00 to 2026-03-11 23:00:00
  Wind speed: 0.4 - 21.2 m/s
  Temperature: -1.6 - 11.5 °C


,Datetime (UTC),wind_speed,temperature
211,2026-03-11 19:00:00,15.1,5.8
212,2026-03-11 20:00:00,12.2,4.8
213,2026-03-11 21:00:00,13.0,3.3
214,2026-03-11 22:00:00,12.6,2.4
215,2026-03-11 23:00:00,11.9,1.9


# Fetch Carbon Intensity History (Energi Data Service)

CO2 emissions data from Energinet — updated every 5 minutes, free, no API key.
We fetch the last 8 days of data and resample to hourly.


In [5]:
def fetch_carbon_intensity_eds():
    """Fetch past 8 days of carbon intensity from Energi Data Service (CO2Emis dataset)."""
    url = "https://api.energidataservice.dk/dataset/CO2Emis"
    params = {
        "start": "now-P8D",
        "filter": json.dumps({"PriceArea": ["DK1", "DK2"]}),
        "sort": "Minutes5UTC asc",
        "limit": 0,
    }
    
    resp = requests.get(url, params=params, timeout=30)
    resp.raise_for_status()
    records = resp.json().get("records", [])
    
    if not records:
        print("⚠️  No carbon intensity data returned from Energi Data Service")
        return pd.DataFrame()
    
    df = pd.DataFrame(records)
    df["Datetime (UTC)"] = pd.to_datetime(df["Minutes5UTC"])
    
    # Average DK1 + DK2, resample to hourly
    hourly = df.groupby(pd.Grouper(key="Datetime (UTC)", freq="1h"))["CO2Emission"].mean()
    hourly = hourly.reset_index()
    hourly.columns = ["Datetime (UTC)", "carbon_intensity"]
    hourly = hourly.dropna()
    
    print(f"Carbon intensity: {len(hourly)} hours")
    print(f"  Range: {hourly['Datetime (UTC)'].min()} to {hourly['Datetime (UTC)'].max()}")
    print(f"  Values: {hourly['carbon_intensity'].min():.1f} - {hourly['carbon_intensity'].max():.1f} gCO₂eq/kWh")
    
    return hourly

carbon_df = fetch_carbon_intensity_eds()
carbon_df.tail()


Carbon intensity: 193 hours
  Range: 2026-03-02 20:00:00 to 2026-03-10 20:00:00
  Values: 39.8 - 189.3 gCO₂eq/kWh


,Datetime (UTC),carbon_intensity
188,2026-03-10 16:00:00,77.833333
189,2026-03-10 17:00:00,73.375000
190,2026-03-10 18:00:00,79.000000
191,2026-03-10 19:00:00,82.583333
192,2026-03-10 20:00:00,72.687500


# Fetch Day-Ahead Electricity Prices (Energi Data Service)

Day-ahead prices are published daily around 13:00 CET for the next day.
We fetch the last 8 days of prices from the Elspotprices dataset.


In [9]:
import requests, json

url = "https://api.energidataservice.dk/dataset/Elspotprices"
params = {
    "start": "now-P1D",
    "filter": json.dumps({"PriceArea": ["DK1"]}),
    "sort": "HourUTC asc",
    "limit": 5,
}

resp = requests.get(url, params=params, timeout=30)
print(f"Status: {resp.status_code}")
data = resp.json()
print(f"Total: {data.get('total')}")
print(f"Records: {len(data.get('records', []))}")

if data.get("records"):
    print(f"\nColumns: {list(data['records'][0].keys())}")
    print(f"\nFirst record:")
    for k, v in data['records'][0].items():
        print(f"  {k}: {v}")
else:
    # Try without the columns filter to see what's available
    print("\nTrying without columns filter...")
    params2 = {
        "start": "now-P1D",
        "limit": 2,
    }
    resp2 = requests.get(url, params=params2, timeout=30)
    data2 = resp2.json()
    print(f"Total: {data2.get('total')}")
    if data2.get("records"):
        print(f"Columns: {list(data2['records'][0].keys())}")

Status: 200
Total: 0
Records: 0

Trying without columns filter...
Total: 0


# Build Input Tensor

Merge all data sources into a single DataFrame, compute derived features, 
normalize using the saved scaler, and extract the most recent 168-hour window.


In [7]:
def build_input_tensor(carbon_df, weather_df, price_df):
    """
    Merge all data, compute features, normalize, and return 
    the input tensor ready for the model.
    """
    # ─── Merge on Datetime (UTC) ───
    merged = carbon_df.copy()
    merged = merged.merge(weather_df, on="Datetime (UTC)", how="left")
    merged = merged.merge(price_df, on="Datetime (UTC)", how="left")
    
    # Forward-fill small gaps (e.g. if weather data has a missing hour)
    merged = merged.sort_values("Datetime (UTC)").reset_index(drop=True)
    merged["wind_speed"] = merged["wind_speed"].ffill()
    merged["temperature"] = merged["temperature"].ffill()
    merged["price_eur_mwh"] = merged["price_eur_mwh"].ffill()
    
    # ─── Compute derived features ───
    dt = merged["Datetime (UTC)"]
    merged["hour_sin"]  = np.sin(2 * np.pi * dt.dt.hour / 24)
    merged["hour_cos"]  = np.cos(2 * np.pi * dt.dt.hour / 24)
    merged["year_sin"]  = np.sin(2 * np.pi * dt.dt.dayofyear / 365.25)
    merged["year_cos"]  = np.cos(2 * np.pi * dt.dt.dayofyear / 365.25)
    merged["dow_sin"]   = np.sin(2 * np.pi * dt.dt.dayofweek / 7)
    merged["dow_cos"]   = np.cos(2 * np.pi * dt.dt.dayofweek / 7)
    merged["diff_1"]    = merged["carbon_intensity"].diff(1)
    merged["diff_24"]   = merged["carbon_intensity"].diff(24)
    
    # Drop NaN rows from differencing
    merged = merged.dropna().reset_index(drop=True)
    
    # ─── Check we have enough data ───
    if len(merged) < WINDOW_SIZE:
        raise ValueError(f"Not enough data: have {len(merged)} rows, need {WINDOW_SIZE}")
    
    # ─── Extract the most recent WINDOW_SIZE hours ───
    recent = merged.tail(WINDOW_SIZE).copy()
    last_timestamp = recent["Datetime (UTC)"].iloc[-1]
    
    print(f"Input window: {recent['Datetime (UTC)'].iloc[0]} to {last_timestamp}")
    print(f"  Rows: {len(recent)}")
    
    # ─── Normalize using saved scaler parameters ───
    feature_values = recent[FEATURE_NAMES].values.astype(np.float32)
    feature_values_scaled = (feature_values - feature_means) / feature_scales
    
    # Reshape to (1, 168, 12) for single-sample prediction
    input_tensor = feature_values_scaled.reshape(1, WINDOW_SIZE, len(FEATURE_NAMES))
    
    print(f"  Input tensor shape: {input_tensor.shape}")
    print(f"  Feature ranges (scaled): min={input_tensor.min():.2f}, max={input_tensor.max():.2f}")
    
    return input_tensor, last_timestamp

input_tensor, last_timestamp = build_input_tensor(carbon_df, weather_df, price_df)


KeyError: 'Datetime (UTC)'

# Run Prediction

In [ ]:
# ─── Run the model ───
pred_scaled = model.predict(input_tensor)[0]

# ─── Inverse transform to gCO₂eq/kWh ───
pred_values = pred_scaled * target_scale + target_mean

# ─── Build forecast timestamps ───
forecast_times = [last_timestamp + timedelta(hours=h+1) for h in range(FORECAST_HORIZON)]

forecast_df = pd.DataFrame({
    "Datetime (UTC)": forecast_times,
    "Hour Ahead": range(1, FORECAST_HORIZON + 1),
    "Predicted gCO₂eq/kWh": pred_values.round(1),
})

print(f"\n24-Hour Carbon Intensity Forecast")
print(f"Generated at: {last_timestamp}")
print(f"Forecast period: {forecast_times[0]} to {forecast_times[-1]}")
print()
print(forecast_df.to_string(index=False))


# Visualize Forecast

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

hours = forecast_df["Hour Ahead"]
values = forecast_df["Predicted gCO₂eq/kWh"]

# Color bars by intensity (green=low carbon, red=high carbon)
colors = plt.cm.RdYlGn_r((values - values.min()) / (values.max() - values.min() + 1e-6))

ax.bar(hours, values, color=colors, edgecolor="white", linewidth=0.5)

# Add value labels on bars
for h, v in zip(hours, values):
    ax.text(h, v + 1, f"{v:.0f}", ha="center", va="bottom", fontsize=8)

# Add time labels on x-axis
time_labels = [t.strftime("%H:%M") for t in forecast_times]
ax.set_xticks(hours)
ax.set_xticklabels(time_labels, rotation=45, ha="right", fontsize=8)

ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Carbon Intensity (gCO₂eq/kWh)")
ax.set_title(f"24-Hour Carbon Intensity Forecast for Denmark\n"
             f"Generated: {last_timestamp.strftime('%Y-%m-%d %H:%M UTC')}")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()


# Find Optimal Charging Window

In [ ]:
def find_best_charging_window(forecast_df, window_hours=3):
    """Find the window_hours consecutive hours with lowest average carbon intensity."""
    values = forecast_df["Predicted gCO₂eq/kWh"].values
    times = forecast_df["Datetime (UTC)"].values
    
    best_avg = float("inf")
    best_start = 0
    
    for i in range(len(values) - window_hours + 1):
        avg = values[i:i+window_hours].mean()
        if avg < best_avg:
            best_avg = avg
            best_start = i
    
    best_end = best_start + window_hours
    worst_avg = max(
        values[i:i+window_hours].mean() 
        for i in range(len(values) - window_hours + 1)
    )
    
    savings = worst_avg - best_avg
    
    return {
        "start_hour": best_start + 1,
        "end_hour": best_end,
        "start_time": pd.Timestamp(times[best_start]),
        "end_time": pd.Timestamp(times[best_end - 1]) + timedelta(hours=1),
        "avg_intensity": best_avg,
        "worst_avg": worst_avg,
        "savings_gco2": savings,
    }

# Find best 3-hour window
result = find_best_charging_window(forecast_df, window_hours=3)

print(f"🔋 Best 3-Hour Charging Window")
print(f"━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print(f"  Start: {result['start_time'].strftime('%Y-%m-%d %H:%M UTC')} (hour {result['start_hour']})")
print(f"  End:   {result['end_time'].strftime('%Y-%m-%d %H:%M UTC')} (hour {result['end_hour']})")
print(f"  Avg carbon intensity: {result['avg_intensity']:.1f} gCO₂eq/kWh")
print(f"  Worst 3h window:      {result['worst_avg']:.1f} gCO₂eq/kWh")
print(f"  Savings vs worst:     {result['savings_gco2']:.1f} gCO₂eq/kWh ({result['savings_gco2']/result['worst_avg']*100:.0f}%)")

# ─── Plot with charging window highlighted ───
fig, ax = plt.subplots(figsize=(14, 6))

values = forecast_df["Predicted gCO₂eq/kWh"].values
hours = forecast_df["Hour Ahead"].values
time_labels = [t.strftime("%H:%M") for t in forecast_times]

ax.plot(hours, values, "o-", linewidth=2, markersize=6, color="steelblue")

# Highlight best window
ax.axvspan(result["start_hour"] - 0.5, result["end_hour"] + 0.5, 
           alpha=0.2, color="green", label=f"Best charging: {result['start_time'].strftime('%H:%M')}-{result['end_time'].strftime('%H:%M')}")

ax.set_xticks(hours)
ax.set_xticklabels(time_labels, rotation=45, ha="right", fontsize=8)
ax.set_xlabel("Time (UTC)")
ax.set_ylabel("Carbon Intensity (gCO₂eq/kWh)")
ax.set_title("24h Forecast with Optimal Charging Window")
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


# Save Forecast Locally (for testing)

In [ ]:
# Save the forecast to a local JSON file (simulates what the app would cache)
output = {
    "generated_at": last_timestamp.isoformat(),
    "forecast_period": {
        "start": forecast_times[0].isoformat(),
        "end": forecast_times[-1].isoformat(),
    },
    "best_charging_window": {
        "start": result["start_time"].isoformat(),
        "end": result["end_time"].isoformat(),
        "avg_carbon_intensity": round(result["avg_intensity"], 1),
    },
    "hourly_forecast": [
        {
            "time": t.isoformat(),
            "hour_ahead": h,
            "carbon_intensity_gco2_kwh": round(v, 1),
        }
        for t, h, v in zip(forecast_times, range(1, 25), pred_values)
    ],
}

with open("latest_forecast.json", "w") as f:
    json.dump(output, f, indent=2, default=str)

print("Forecast saved to latest_forecast.json")
print(f"\nJSON preview:")
print(json.dumps(output, indent=2, default=str)[:800])
